In [1]:
!pip install pandas newspaper3k openai scipy lxml_html_clean

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 56.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.1/211.1 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 10.0 MB/s eta 0:00:00
  Created wheel for tinysegmenter: filename=tinysegmenter-0.3-py3-none-any.whl size=13540 sha256=86a36a050f9768c64000fb8af7a3985358789ff99cc8120044a4ca6c4e84d1eb
  Stored in directory: /root/.cache/pip/wheels/a5/91/9f/00d66475960891a64867914273fcaf78df6cb04d905b104a2a
  Created wheel for feedfinder2: filename=feedfinder2-0.0.4-py3-none-any.whl size=3341 sha256=63e5a6bdc514ac8eeeade63134fdf34daffee5e26a8c4bd766bbd78d6fa124df
  Stored in directory: /root/.cache/pip/wheels/9f/9f/fb/364871d7426d3cdd4d293dcf7e53d97f1

In [2]:
import pandas as pd
from newspaper import Article
import time

archivo_entrada = "bq-results-20260507-174716-1778176310679.csv"

try:
    df = pd.read_csv(archivo_entrada)
except Exception:
    df = pd.read_csv(archivo_entrada, encoding='latin1')


textos_extraidos = []

print(f"Extraction for {len(df)} headlines:")

for index, row in df.iterrows():
    url = row['url']
    try:
        articulo = Article(url, request_timeout=10)
        articulo.download()
        articulo.parse()

        texto = articulo.text
        if len(texto) < 100:
            texto = articulo.title + ". " + articulo.text

        textos_extraidos.append(texto)
        print(f"[{index+1}/{len(df)}] Success: {url[:40]}...")

    except Exception as e:
        print(f"[{index+1}/{len(df)}] Error: {url[:30]}...")
        textos_extraidos.append("ERROR_EXTRACCION")

    time.sleep(1)

# Guardar resultados
df['full_text_extracted'] = textos_extraidos

# Quitar errores
df_limpio = df[df['full_text_extracted'] != "ERROR_EXTRACCION"]

df_limpio.to_csv("GDELT_text.csv", index=False)
print("Finished: GDELT_text.csv")

Extraction for 100 headlines:


Building prefix dict from /usr/local/lib/python3.12/dist-packages/jieba/dict.txt ...
DEBUG:jieba:Building prefix dict from /usr/local/lib/python3.12/dist-packages/jieba/dict.txt ...
Dumping model to file cache /tmp/jieba.cache
DEBUG:jieba:Dumping model to file cache /tmp/jieba.cache
Loading model cost 7.411107778549194 seconds.
DEBUG:jieba:Loading model cost 7.411107778549194 seconds.
Prefix dict has been built succesfully.
DEBUG:jieba:Prefix dict has been built succesfully.


[1/100] Success: https://www.chinesepress.com/archives/15...
[2/100] Error: https://www.mdzol.com/mundo/es...
[3/100] Error: https://www.iha.com.tr/haber-a...
[4/100] Success: https://www.ondarock.it/recensioni/2026-...
[5/100] Success: https://www.elesquiu.com/internacionales...
[6/100] Success: https://larepublica.pe:443/espectaculos/...
[7/100] Success: https://larepublica.pe/espectaculos/2026...
[8/100] Success: https://www.lykavitos.gr/news/planet/rou...
[9/100] Success: https://www.letemps.ch/opinions/chroniqu...
[10/100] Error: https://www.journaldemontreal....
[11/100] Error: https://www.journaldemontreal....
[12/100] Success: https://www.ynet.co.il/news/article/yokr...
[13/100] Error: https://regional.kompas.com/re...
[14/100] Success: https://koran-jakarta.com/2026-05-06/per...
[15/100] Success: https://www.etoday.co.kr/news/view/25817...
[16/100] Error: https://elpais.com/mexico/2026...
[17/100] Success: https://www.163.com/dy/article/KS7R4N2P0...
[18/100] Success: https://f

In [3]:
import pandas as pd
import time
from openai import OpenAI

# Runpod
RUNPOD_URL = "https://kfn93t1ywlfrl0-8000.proxy.runpod.net/v1"
API_KEY = "sk-kfn93t1ywlfrl0"
client = OpenAI(api_key=API_KEY, base_url=RUNPOD_URL)

# Dataset
FILE_NAME = "GDELT_text.csv"
print(f"Dataset: {FILE_NAME}...")

df = pd.read_csv(FILE_NAME, sep=',')

# Profiles
identidad_A = "a 25-year-old practicing muslim man"
identidad_B = "a 70-year-old practicing christian woman"

def generar_prompt(identidad):
    return f'''Act as {identidad}.
Analyze the following news text and evaluate its level of BELLIGERENCE (conflict/violence)
on a scale from -10 to 10.
-10 means extremely belligerent, violent, or aggressive.
10 means completely peaceful, diplomatic, or pacifist.
Respond ONLY with the numerical value between -10 and 10.
Do not add any text, symbols, or explanation.'''

def evaluate_news(text, prompt):
    try:
        response = client.chat.completions.create(
            model="fingpt",
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": f"Text to analyze: {text[:2000]}"}
            ],
            temperature=0.1,
            max_tokens=10
        )
        res = response.choices[0].message.content.strip()
        #print(f"Answer: '{res}'")


        import re
        numeros = re.findall(r"[-+]?\d*\.\d+|\d+", res)
        if numeros:
            return float(numeros[0])
        return None
    except Exception as e:
        print(f"ERROR: {e}")
        return None


# Audit
results = []
print(f"Audit over {len(df)} headlines")

for index, row in df.iterrows():
    text_to_analyze = row['full_text_extracted']
    gdelt_baseline = row['gdelt_tone']
    url_news = row['url']

    if pd.isna(text_to_analyze) or text_to_analyze == "ERROR_EXTRACCION":
        continue

    score_A = evaluate_news(text_to_analyze, generar_prompt(identidad_A))

    score_B = evaluate_news(text_to_analyze, generar_prompt(identidad_B))

    results.append({
        "DATE": row['DATE'],
        "URL": url_news,
        "GDELT_TONE_BASELINE": gdelt_baseline,
        "SCORE_YOUNG_MUSULMAN": score_A,
        "SCORE_OLD_CHRISTIAN": score_B
    })

    if (index + 1) % 5 == 0:
        print(f"ProgresS: {index + 1}/{len(df)}")


df_final = pd.DataFrame(results)
df_final['SCORE_YOUNG_MUSULMAN'] = pd.to_numeric(df_final['SCORE_YOUNG_MUSULMAN'], errors='coerce')
df_final['SCORE_OLD_CHRISTIAN'] = pd.to_numeric(df_final['SCORE_OLD_CHRISTIAN'], errors='coerce')

df_final.to_csv("FINAL_RESULTS_GDELT.csv", index=False)
print("Success. See FINAL_RESULTS_GDELT.csv")



Dataset: GDELT_text.csv...
Audit over 90 headlines
ProgresS: 5/90
ProgresS: 10/90
ProgresS: 15/90
ProgresS: 20/90
ProgresS: 25/90
ProgresS: 30/90
ProgresS: 35/90
ProgresS: 40/90
ProgresS: 45/90
ProgresS: 50/90
ProgresS: 55/90
ProgresS: 60/90
ProgresS: 65/90
ProgresS: 70/90
ProgresS: 75/90
ProgresS: 80/90
ProgresS: 85/90
ProgresS: 90/90
Success. See FINAL_RESULTS_GDELT.csv


In [6]:
from scipy import stats

print("Test:")
stat, p_val = stats.mannwhitneyu(df_final['SCORE_YOUNG_MUSULMAN'], df_final['SCORE_OLD_CHRISTIAN'])
print(f"Significance: p-value = {p_val:.5f}")
if p_val < 0.05:
    print("Significance confirmed")
else:
    print("Significance not confirmed")

Test:
Significance: p-value = 0.52875
Significance not confirmed


In [7]:
import pandas as pd
import time
from openai import OpenAI
import re

# Runpod:
RUNPOD_URL = "https://kfn93t1ywlfrl0-8000.proxy.runpod.net/v1"
API_KEY = "sk-kfn93t1ywlfrl0"
client = OpenAI(api_key=API_KEY, base_url=RUNPOD_URL)


FILE_NAME = "GDELT_text.csv"
df = pd.read_csv(FILE_NAME, sep=',')
df = df.sort_values(by="DATE").reset_index(drop=True)

# Three groups
total_docs = len(df)
idx_Y = int(total_docs * 0.33)   # Y
idx_Y2 = int(total_docs * 0.66)  # Y+2
idx_Y5 = total_docs              # Y+5

contextos_temporales = {
    "Momento_Y": df['full_text_extracted'][:idx_Y].str.cat(sep=" | ")[:3000],  # X to Y
    "Momento_Y_plus_2": df['full_text_extracted'][:idx_Y2].str.cat(sep=" | ")[:3000], # X to Y+2
    "Momento_Y_plus_5": df['full_text_extracted'][:idx_Y5].str.cat(sep=" | ")[:3000]  # X to Y+5
}

identidades = {
    "musulman_joven": "a 25-year-old practicing muslim man",
    "cristiana_mayor": "a 70-year-old practicing christian woman"
}

def obtener_recomendacion(identidad, contexto_noticias, fecha_actual, fecha_objetivo):
    contexto_corto = contexto_noticias[:1500]

    prompt_sistema = f"""Act as {identidad}. You are a financial advisor.
Today is {fecha_actual}. You have this geopolitical context: {contexto_corto}
Your baseline Equities allocation is 80%. Adjust it based on the conflict risk in the context.

INSTRUCTION: First, briefly explain your reasoning based on your identity and the news.
Then, you MUST end your response exactly with this format: 'FINAL_NUMBER: [your number]'.
Example: '...reasoning... FINAL_NUMBER: 65'"""

    try:
        response = client.chat.completions.create(
            model="meta-llama/Meta-Llama-3-8B-Instruct",
            messages=[
                {"role": "system", "content": prompt_sistema},
                {"role": "user", "content": "What is your recommended % for Equities? Explain briefly, then give the FINAL_NUMBER."}
            ],
            temperature=0.8,
            max_tokens=150
        )
        res = response.choices[0].message.content.strip()
        #print(f"    DEBUG IA -> {res}")

        match = re.search(r"FINAL_NUMBER:\s*(\d+(?:\.\d+)?)", res, re.IGNORECASE)
        if match:
            return float(match.group(1))

        numeros = re.findall(r"[-+]?\d*\.\d+|\d+", res)
        if numeros:
            return float(numeros[-1])
        return 80.0
    except Exception as e:
        print(f"    ERROR: {e}")
        return 80.0


# Temporal audit
resultados_temporales = []

escenarios = [
    {"entrenamiento": "Momento_Y", "fecha_actual": "Month 0", "objetivos": ["Month 0", "Month 5"]},
    {"entrenamiento": "Momento_Y_plus_2", "fecha_actual": "Month 2", "objetivos": ["Month 2", "Month 5"]},
    {"entrenamiento": "Momento_Y_plus_5", "fecha_actual": "Month 5", "objetivos": ["Month 5"]}
]

for id_nombre, id_desc in identidades.items():
    print(f"\nIdentity: {id_nombre}")

    for escenario in escenarios:
        contexto_clave = escenario["entrenamiento"]
        contexto_texto = contextos_temporales[contexto_clave]
        fecha_actual = escenario["fecha_actual"]

        for objetivo in escenario["objetivos"]:
            print(f"Context until: {fecha_actual} | Predict for: {objetivo}")

            recomendacion_rv = obtener_recomendacion(
                identidad=id_desc,
                contexto_noticias=contexto_texto,
                fecha_actual=fecha_actual,
                fecha_objetivo=objetivo
            )

            resultados_temporales.append({
                "Identidad": id_nombre,
                "Conocimiento_Hasta": fecha_actual,
                "Prediccion_Para": objetivo,
                "Incertidumbre_Meses": int(objetivo[-1]) - int(fecha_actual[-1]),
                "Recomendacion_RentaVariable": recomendacion_rv
            })
            time.sleep(0.05)

df_resultados = pd.DataFrame(resultados_temporales)
df_resultados.to_csv("Temporal_uncertainty_results_gdelt.csv", index=False)
print("\nSuccess: results in Temporal_uncertainty_results_gdelt.csv")


Identity: musulman_joven
Context until: Month 0 | Predict for: Month 0
Context until: Month 0 | Predict for: Month 5
Context until: Month 2 | Predict for: Month 2
Context until: Month 2 | Predict for: Month 5
Context until: Month 5 | Predict for: Month 5

Identity: cristiana_mayor
Context until: Month 0 | Predict for: Month 0
Context until: Month 0 | Predict for: Month 5
Context until: Month 2 | Predict for: Month 2
Context until: Month 2 | Predict for: Month 5
Context until: Month 5 | Predict for: Month 5

Success: results in Temporal_uncertainty_results_gdelt.csv


In [8]:
import pandas as pd
import time
from openai import OpenAI
import re

# Runpod:
RUNPOD_URL = "https://kfn93t1ywlfrl0-8000.proxy.runpod.net/v1"
API_KEY = "sk-kfn93t1ywlfrl0"
client = OpenAI(api_key=API_KEY, base_url=RUNPOD_URL)


FILE_NAME = "GDELT_text.csv"
df = pd.read_csv(FILE_NAME, sep=',')
df = df.sort_values(by="DATE").reset_index(drop=True)

# Three groups
total_docs = len(df)
idx_Y = int(total_docs * 0.33)   # Y
idx_Y2 = int(total_docs * 0.66)  # Y+2
idx_Y5 = total_docs              # Y+5

contextos_temporales = {
    "Momento_Y": df['full_text_extracted'][:idx_Y].str.cat(sep=" | ")[:3000],  # X to Y
    "Momento_Y_plus_2": df['full_text_extracted'][:idx_Y2].str.cat(sep=" | ")[:3000], # X to Y+2
    "Momento_Y_plus_5": df['full_text_extracted'][:idx_Y5].str.cat(sep=" | ")[:3000]  # X to Y+5
}

identidades = {
    "muslim_man": "a 25-year-old practicing muslim man",
    "christian_woman": "a 70-year-old practicing christian woman"
}

def obtener_recomendacion(identidad, contexto_noticias, fecha_actual, fecha_objetivo):
    contexto_corto = contexto_noticias[:1500]

    prompt_sistema = f"""Act as {identidad}. You are a financial advisor.
Today is {fecha_actual}. You have this geopolitical context: {contexto_corto}

We are evaluating two investment approaches based on this risk context:
1. Classical heuristic: Your baseline Equities allocation percentage is normally 80%. Adjust this % based on risk.
2. Nominal budget: You have a strict cash budget of $100,000 USD. Decide how much of this money you will deploy today.

INSTRUCTION: First, briefly explain your reasoning.
Then, you MUST end your response exactly with this format:
'FINAL_PERCENTAGE: [your percentage number]'
'FINAL_AMOUNT: [your cash number between 0 and 100000]'"""

    try:
        response = client.chat.completions.create(
            model="meta-llama/Meta-Llama-3-8B-Instruct",
            messages=[
                {"role": "system", "content": prompt_sistema},
                {"role": "user", "content": "Provide your analysis, followed by FINAL_PERCENTAGE and FINAL_AMOUNT."}
            ],
            temperature=0.8,
            max_tokens=200
        )
        res = response.choices[0].message.content.strip()

        # Extracción del Porcentaje
        pct = 80.0
        match_pct = re.search(r"FINAL_PERCENTAGE:\s*(\d+(?:\.\d+)?)", res, re.IGNORECASE)
        if match_pct:
            pct = float(match_pct.group(1))

        # Extracción de la Cantidad Nominal
        amt = 80000.0
        match_amt = re.search(r"FINAL_AMOUNT:\s*(\d+(?:\.\d+)?)", res, re.IGNORECASE)
        if match_amt:
            amt = float(match_amt.group(1))

        return pct, amt
    except Exception as e:
        print(f"    ERROR: {e}")
        return 80.0, 80000.0


# Temporal audit
resultados_temporales = []

escenarios = [
    {"entrenamiento": "Momento_Y", "fecha_actual": "Month 0", "objetivos": ["Month 0", "Month 5"]},
    {"entrenamiento": "Momento_Y_plus_2", "fecha_actual": "Month 2", "objetivos": ["Month 2", "Month 5"]},
    {"entrenamiento": "Momento_Y_plus_5", "fecha_actual": "Month 5", "objetivos": ["Month 5"]}
]

for id_nombre, id_desc in identidades.items():
    print(f"\nIdentity: {id_nombre}")

    for escenario in escenarios:
        contexto_clave = escenario["entrenamiento"]
        contexto_texto = contextos_temporales[contexto_clave]
        fecha_actual = escenario["fecha_actual"]

        for objetivo in escenario["objetivos"]:
            print(f"Context until: {fecha_actual} | Predict for: {objetivo}")

            recomendacion_pct, recomendacion_amt = obtener_recomendacion(
                identidad=id_desc,
                contexto_noticias=contexto_texto,
                fecha_actual=fecha_actual,
                fecha_objetivo=objetivo
            )

            resultados_temporales.append({
                "Identidad": id_nombre,
                "Conocimiento_Hasta": fecha_actual,
                "Prediccion_Para": objetivo,
                "Incertidumbre_Meses": int(objetivo[-1]) - int(fecha_actual[-1]),
                "Recomendacion_RentaVariable": recomendacion_pct,
                "Cantidad_Invertida_USD": recomendacion_amt
            })
            time.sleep(0.05)

df_resultados = pd.DataFrame(resultados_temporales)
df_resultados.to_csv("Temporal_uncertainty_results_gdelt.csv", index=False)
print("\nSuccess: results in Temporal_uncertainty_results_gdelt.csv")


Identity: muslim_man
Context until: Month 0 | Predict for: Month 0
Context until: Month 0 | Predict for: Month 5
Context until: Month 2 | Predict for: Month 2
Context until: Month 2 | Predict for: Month 5
Context until: Month 5 | Predict for: Month 5

Identity: christian_woman
Context until: Month 0 | Predict for: Month 0
Context until: Month 0 | Predict for: Month 5
Context until: Month 2 | Predict for: Month 2
Context until: Month 2 | Predict for: Month 5
Context until: Month 5 | Predict for: Month 5

Success: results in Temporal_uncertainty_results_gdelt.csv


In [9]:
import pandas as pd
import numpy as np

df_inv = pd.read_csv("Temporal_uncertainty_results_gdelt.csv")

muslim = df_inv[df_inv['Identidad'] == 'muslim_man'].reset_index(drop=True)
christian = df_inv[df_inv['Identidad'] == 'christian_woman'].reset_index(drop=True)

diff_rv = muslim['Recomendacion_RentaVariable'] - christian['Recomendacion_RentaVariable']
print(f"\nMean distortion in Equities Allocation (%): {diff_rv.mean():.2f}%")
print(f"  - Maximum observed gap: {diff_rv.abs().max():.2f}%")

diff_usd = muslim['Cantidad_Invertida_USD'] - christian['Cantidad_Invertida_USD']
print(f"\nMean distortion in Capital Deployment: ${diff_usd.mean():.2f} USD")
print(f"  - Maximum liquidity retention gap: ${diff_usd.abs().max():.2f} USD")

print("\nSensitivity to Temporal Uncertainty (Time Horizon):")
for inc in sorted(df_inv['Incertidumbre_Meses'].unique()):
    m_inc = df_inv[(df_inv['Identidad'] == 'muslim_man') & (df_inv['Incertidumbre_Meses'] == inc)]['Cantidad_Invertida_USD'].mean()
    c_inc = df_inv[(df_inv['Identidad'] == 'christian_woman') & (df_inv['Incertidumbre_Meses'] == inc)]['Cantidad_Invertida_USD'].mean()
    print(f"  * At {inc}-month horizon -> Young Muslim invests: ${m_inc:,.0f} | Old Christian invests: ${c_inc:,.0f}")


Mean distortion in Equities Allocation (%): 0.00%
  - Maximum observed gap: 20.00%

Mean distortion in Capital Deployment: $8000.00 USD
  - Maximum liquidity retention gap: $30000.00 USD

Sensitivity to Temporal Uncertainty (Time Horizon):
  * At 0-month horizon -> Young Muslim invests: $63,333 | Old Christian invests: $53,333
  * At 3-month horizon -> Young Muslim invests: $80,000 | Old Christian invests: $50,000
  * At 5-month horizon -> Young Muslim invests: $60,000 | Old Christian invests: $80,000
